<a href="https://colab.research.google.com/github/FernandoJavierNegro/Prueba-2026/blob/main/PRUEBA2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# EfficientDet-Lite1 para detectar botellas

Este flujo en Google Colab toma un .zip con imágenes y etiquetas YOLO .txt, elimina pares inválidos, divide las imágenes en train, validation y test, genera el CSV requerido por TensorFlow Lite Model Maker, entrena EfficientDet-Lite1, exporta un modelo TFLite Float16 con metadatos y descarga un paquete preparado para Android.

EfficientDet-Lite1 está diseñado para dispositivos móviles y ofrece un equilibrio mayor de precisión que Lite0, aunque con algo más de latencia. Model Maker admite datasets mediante CSV y puede exportar el detector con las etiquetas incorporadas en los metadatos; estos modelos son compatibles con la API ObjectDetector de TensorFlow Lite Task Library para Android.

En Colab active primero: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.

Estructura admitida del ZIP

Puede tener cualquiera de estas estructuras:

dataset/
├── images/
│   ├── foto1.jpg
│   └── foto2.jpg
├── labels/
│   ├── foto1.txt
│   └── foto2.txt
└── data.yaml

También admite subcarpetas:

dataset/
├── images/train/
├── images/val/
├── labels/train/
└── labels/val/

Cada TXT debe estar en formato YOLO:

class_id x_center y_center width height

Para una sola clase:

0 0.5321 0.4812 0.2843 0.6115


## Celda 1 — Instalar dependencias

La documentación oficial de Model Maker todavía utiliza una pila de dependencias antigua. Por eso conviene ejecutar esta celda en un entorno Colab limpio y reiniciar la sesión cuando termine.


In [ ]:
# ============================================================
# INSTALAR TENSORFLOW LITE MODEL MAKER
# ============================================================

!apt-get update -qq
!apt-get install -y -qq libportaudio2

!pip install -q --upgrade pip setuptools wheel
!pip install -q pycocotools
!pip install -q tflite-model-maker
!pip install -q scikit-learn pandas pillow pyyaml

print("✅ Instalación finalizada.")
print("Reinicie el entorno de ejecución antes de continuar:")
print("Entorno de ejecución → Reiniciar sesión")



Después del reinicio, continúe desde la siguiente celda.


## Celda 2 — Importaciones y configuración


In [ ]:
# ============================================================
# IMPORTACIONES Y CONFIGURACIÓN
# ============================================================

import csv
import gc
import json
import os
import random
import shutil
import zipfile

from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import yaml

from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from google.colab import files

from tflite_model_maker import model_spec
from tflite_model_maker import object_detector

from tflite_model_maker.config import QuantizationConfig
from tflite_model_maker.config import ExportFormat


# ============================================================
# SEMILLA
# ============================================================

SEMILLA = 42

random.seed(SEMILLA)
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)


# ============================================================
# CONFIGURACIÓN DEL PROYECTO
# ============================================================

NOMBRES_CLASES = {
    0: "botellas"
}

PORCENTAJE_TRAIN = 0.70
PORCENTAJE_VALIDATION = 0.20
PORCENTAJE_TEST = 0.10

EPOCHS = 60

# EfficientDet-Lite1 requiere más memoria que Lite0.
# Cambiar a 4 o 2 si aparece ResourceExhaustedError.
BATCH_SIZE = 8

TRAIN_WHOLE_MODEL = True


# ============================================================
# CARPETAS
# ============================================================

BASE_DIR = Path("/content/efficientdet_lite1_botellas")

DATASET_EXTRAIDO = BASE_DIR / "dataset_extraido"
DATASET_PREPARADO = BASE_DIR / "dataset_preparado"

IMAGES_DIR = DATASET_PREPARADO / "images"

EXPORT_DIR = BASE_DIR / "modelo_exportado"
RESULTADOS_DIR = BASE_DIR / "resultados"

CSV_MODELO = DATASET_PREPARADO / "annotations_model_maker.csv"
CSV_LECTURA = DATASET_PREPARADO / "annotations_legible.csv"

MODELO_FP16 = EXPORT_DIR / "efficientdet_lite1_botellas_fp16.tflite"
MODELO_INT8 = EXPORT_DIR / "efficientdet_lite1_botellas_int8.tflite"

PAQUETE_ANDROID = BASE_DIR / "paquete_android"
ZIP_FINAL = Path("/content/efficientdet_lite1_botellas_android.zip")


# ============================================================
# LIMPIAR EJECUCIONES ANTERIORES
# ============================================================

if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)

if ZIP_FINAL.exists():
    ZIP_FINAL.unlink()

for carpeta in [
    DATASET_EXTRAIDO,
    DATASET_PREPARADO,
    IMAGES_DIR,
    EXPORT_DIR,
    RESULTADOS_DIR,
    PAQUETE_ANDROID,
]:
    carpeta.mkdir(parents=True, exist_ok=True)


# ============================================================
# VERIFICAR TENSORFLOW Y GPU
# ============================================================

print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("✅ GPU detectada:", gpus)
else:
    print("⚠ No se detectó GPU. El entrenamiento será más lento.")


## Celda 3 — Subir el ZIP


In [ ]:
# ============================================================
# SUBIR DATASET YOLO EN ZIP
# ============================================================

print("Seleccione el archivo ZIP que contiene imágenes y labels YOLO.\n")

archivos_subidos = files.upload()

if not archivos_subidos:
    raise RuntimeError("No se seleccionó ningún archivo.")

ruta_zip = None

for nombre_archivo in archivos_subidos.keys():

    if nombre_archivo.lower().endswith(".zip"):
        ruta_zip = Path("/content") / nombre_archivo
        break

if ruta_zip is None:
    raise ValueError(
        "Debe seleccionar un archivo con extensión .zip"
    )

print(f"✅ ZIP seleccionado: {ruta_zip.name}")


## Celda 4 — Extraer y localizar imágenes y etiquetas


In [ ]:
# ============================================================
# EXTRAER DATASET
# ============================================================

try:

    with zipfile.ZipFile(ruta_zip, "r") as archivo_zip:
        archivo_zip.extractall(DATASET_EXTRAIDO)

except zipfile.BadZipFile as error:
    raise ValueError("El archivo seleccionado no es un ZIP válido.") from error

print("✅ Dataset extraído.")


# ============================================================
# EXTENSIONES ADMITIDAS
# ============================================================

EXTENSIONES_IMAGEN = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
    ".jfif",
}


# ============================================================
# BUSCAR IMÁGENES
# ============================================================

imagenes_encontradas = sorted([
    ruta
    for ruta in DATASET_EXTRAIDO.rglob("*")
    if ruta.is_file()
    and ruta.suffix.lower() in EXTENSIONES_IMAGEN
])


# ============================================================
# BUSCAR ETIQUETAS TXT
# ============================================================

labels_encontrados = sorted([
    ruta
    for ruta in DATASET_EXTRAIDO.rglob("*.txt")
    if ruta.is_file()
])


print(f"Imágenes encontradas: {len(imagenes_encontradas)}")
print(f"Etiquetas TXT encontradas: {len(labels_encontrados)}")

if not imagenes_encontradas:
    raise RuntimeError("No se encontraron imágenes dentro del ZIP.")

if not labels_encontrados:
    raise RuntimeError("No se encontraron etiquetas YOLO .txt.")


## Celda 5 — Asociar cada imagen con su TXT


In [ ]:
# ============================================================
# ÍNDICE DE ETIQUETAS
# ============================================================

indice_labels = {}

for ruta_label in labels_encontrados:

    indice_labels.setdefault(
        ruta_label.stem.lower(),
        []
    ).append(ruta_label)


def obtener_split_original(ruta):
    """
    Detecta train, val, validation o test en la ruta.
    Se usa solo para mejorar la búsqueda del TXT.
    """

    partes = [parte.lower() for parte in ruta.parts]

    if "train" in partes:
        return "train"

    if "validation" in partes:
        return "validation"

    if "valid" in partes:
        return "validation"

    if "val" in partes:
        return "validation"

    if "test" in partes:
        return "test"

    return None


def buscar_label_correspondiente(ruta_imagen):
    """
    Busca la etiqueta TXT asociada a una imagen.
    """

    partes = list(ruta_imagen.parts)
    partes_minusculas = [parte.lower() for parte in partes]

    # Caso habitual:
    # images/foto.jpg -> labels/foto.txt
    if "images" in partes_minusculas:

        posicion = partes_minusculas.index("images")

        partes_label = partes.copy()
        partes_label[posicion] = "labels"

        candidato = Path(*partes_label).with_suffix(".txt")

        if candidato.exists():
            return candidato

    # Imagen y TXT en la misma carpeta
    candidato = ruta_imagen.with_suffix(".txt")

    if candidato.exists():
        return candidato

    # Buscar por nombre base
    candidatos = indice_labels.get(
        ruta_imagen.stem.lower(),
        []
    )

    if not candidatos:
        return None

    split_imagen = obtener_split_original(ruta_imagen)

    # Priorizar etiqueta del mismo split
    for candidato in candidatos:

        if obtener_split_original(candidato) == split_imagen:
            return candidato

    return candidatos[0]


## Celda 6 — Leer y validar etiquetas YOLO


In [ ]:
# ============================================================
# LEER ETIQUETA YOLO
# ============================================================

def leer_label_yolo(ruta_label):
    """
    Lee etiquetas YOLO:
    class_id x_center y_center width height

    Devuelve una lista de cajas válidas.
    """

    cajas = []
    errores = []

    try:

        with open(
            ruta_label,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as archivo:

            lineas = archivo.readlines()

    except OSError as error:
        return [], [str(error)]

    for numero_linea, linea in enumerate(lineas, start=1):

        linea = linea.strip()

        if not linea:
            continue

        partes = linea.split()

        if len(partes) < 5:

            errores.append(
                f"Línea {numero_linea}: se esperaban 5 valores"
            )

            continue

        try:

            class_id = int(float(partes[0]))
            x_center = float(partes[1])
            y_center = float(partes[2])
            box_width = float(partes[3])
            box_height = float(partes[4])

        except ValueError:

            errores.append(
                f"Línea {numero_linea}: valores no numéricos"
            )

            continue

        if class_id not in NOMBRES_CLASES:

            errores.append(
                f"Línea {numero_linea}: clase desconocida {class_id}"
            )

            continue

        if not (
            0 <= x_center <= 1
            and 0 <= y_center <= 1
            and 0 < box_width <= 1
            and 0 < box_height <= 1
        ):

            errores.append(
                f"Línea {numero_linea}: coordenadas fuera de rango"
            )

            continue

        xmin = x_center - box_width / 2
        ymin = y_center - box_height / 2
        xmax = x_center + box_width / 2
        ymax = y_center + box_height / 2

        # Limitar al área normalizada de la imagen
        xmin = max(0.0, min(xmin, 1.0))
        ymin = max(0.0, min(ymin, 1.0))
        xmax = max(0.0, min(xmax, 1.0))
        ymax = max(0.0, min(ymax, 1.0))

        if xmax <= xmin or ymax <= ymin:

            errores.append(
                f"Línea {numero_linea}: caja sin superficie"
            )

            continue

        cajas.append({
            "class_id": class_id,
            "class_name": NOMBRES_CLASES[class_id],

            "x_center": x_center,
            "y_center": y_center,
            "width": box_width,
            "height": box_height,

            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
        })

    return cajas, errores


## Celda 7 — Validar dataset completo


In [ ]:
# ============================================================
# VALIDAR IMÁGENES Y ANOTACIONES
# ============================================================

registros_imagenes = []
errores_dataset = []

imagenes_sin_txt = 0
imagenes_sin_cajas = 0
imagenes_invalidas = 0

for ruta_imagen in imagenes_encontradas:

    ruta_label = buscar_label_correspondiente(ruta_imagen)

    if ruta_label is None:

        imagenes_sin_txt += 1

        errores_dataset.append({
            "imagen": str(ruta_imagen),
            "error": "No se encontró TXT correspondiente",
        })

        continue

    cajas, errores_label = leer_label_yolo(ruta_label)

    for error in errores_label:

        errores_dataset.append({
            "imagen": str(ruta_imagen),
            "label": str(ruta_label),
            "error": error,
        })

    if not cajas:

        imagenes_sin_cajas += 1
        continue

    try:

        with Image.open(ruta_imagen) as imagen:

            imagen.verify()

        with Image.open(ruta_imagen) as imagen:

            ancho_imagen, alto_imagen = imagen.size

        if ancho_imagen <= 0 or alto_imagen <= 0:
            raise ValueError("Dimensiones de imagen no válidas")

    except (
        UnidentifiedImageError,
        OSError,
        ValueError
    ) as error:

        imagenes_invalidas += 1

        errores_dataset.append({
            "imagen": str(ruta_imagen),
            "error": str(error),
        })

        continue

    registros_imagenes.append({
        "image_path": str(ruta_imagen),
        "label_path": str(ruta_label),
        "image_width": ancho_imagen,
        "image_height": alto_imagen,
        "box_count": len(cajas),
        "boxes": cajas,
    })


df_imagenes = pd.DataFrame(registros_imagenes)

print("=" * 60)
print("RESULTADO DE LA VALIDACIÓN")
print("=" * 60)

print(f"Imágenes válidas:             {len(df_imagenes)}")
print(f"Imágenes sin TXT:             {imagenes_sin_txt}")
print(f"Imágenes sin cajas válidas:   {imagenes_sin_cajas}")
print(f"Imágenes inválidas:           {imagenes_invalidas}")

if len(df_imagenes) < 10:

    raise RuntimeError(
        "Se requieren al menos 10 imágenes válidas para "
        "crear train, validation y test."
    )


if errores_dataset:

    pd.DataFrame(errores_dataset).to_csv(
        RESULTADOS_DIR / "errores_dataset.csv",
        index=False,
        encoding="utf-8-sig"
    )


## Celda 8 — Dividir en train, validation y test

La división se hace por imagen, no por bounding box. De esta forma, una misma fotografía nunca aparece simultáneamente en entrenamiento y prueba.


In [ ]:
# ============================================================
# DIVIDIR POR IMAGEN
# ============================================================

indices = np.arange(len(df_imagenes))

indices_train, indices_temporales = train_test_split(
    indices,
    test_size=(
        PORCENTAJE_VALIDATION
        + PORCENTAJE_TEST
    ),
    random_state=SEMILLA,
    shuffle=True,
)

proporcion_test_temporal = (
    PORCENTAJE_TEST
    / (
        PORCENTAJE_VALIDATION
        + PORCENTAJE_TEST
    )
)

indices_validation, indices_test = train_test_split(
    indices_temporales,
    test_size=proporcion_test_temporal,
    random_state=SEMILLA,
    shuffle=True,
)


df_train = df_imagenes.iloc[
    indices_train
].reset_index(drop=True)

df_validation = df_imagenes.iloc[
    indices_validation
].reset_index(drop=True)

df_test = df_imagenes.iloc[
    indices_test
].reset_index(drop=True)


print(f"Train:       {len(df_train)} imágenes")
print(f"Validation:  {len(df_validation)} imágenes")
print(f"Test:        {len(df_test)} imágenes")


## Celda 9 — Copiar imágenes y generar CSV de Model Maker

Model Maker utiliza una fila por objeto, con el split, la ruta de la imagen, la clase y las coordenadas normalizadas de las esquinas superior izquierda e inferior derecha.


In [ ]:
# ============================================================
# COPIAR IMÁGENES Y CREAR FILAS DEL CSV
# ============================================================

filas_model_maker = []
filas_legibles = []

nombres_usados = set()


def generar_nombre_unico(ruta_imagen, contador):

    nombre_original = ruta_imagen.name
    nombre = nombre_original

    if nombre.lower() not in nombres_usados:

        nombres_usados.add(nombre.lower())
        return nombre

    nombre = (
        f"{contador:06d}_"
        f"{ruta_imagen.stem}"
        f"{ruta_imagen.suffix.lower()}"
    )

    nombres_usados.add(nombre.lower())

    return nombre


def agregar_split(dataframe, nombre_split):

    contador_interno = 0

    for _, fila in dataframe.iterrows():

        contador_interno += 1

        ruta_imagen_original = Path(
            fila["image_path"]
        )

        nombre_salida = generar_nombre_unico(
            ruta_imagen_original,
            contador_interno
        )

        ruta_imagen_salida = (
            IMAGES_DIR
            / nombre_salida
        )

        shutil.copy2(
            ruta_imagen_original,
            ruta_imagen_salida
        )

        ruta_absoluta = str(
            ruta_imagen_salida.resolve()
        )

        for numero_box, caja in enumerate(
            fila["boxes"],
            start=1
        ):

            # Formato CSV Open Images requerido por Model Maker:
            #
            # split, ruta, label,
            # xmin, ymin, vacío, vacío,
            # xmax, ymax, vacío, vacío

            filas_model_maker.append([
                nombre_split,
                ruta_absoluta,
                caja["class_name"],
                caja["xmin"],
                caja["ymin"],
                "",
                "",
                caja["xmax"],
                caja["ymax"],
                "",
                "",
            ])

            filas_legibles.append({
                "split": nombre_split,
                "filename": nombre_salida,
                "image_path": ruta_absoluta,

                "image_width": fila["image_width"],
                "image_height": fila["image_height"],

                "bounding_box_number": numero_box,
                "class_id": caja["class_id"],
                "class_name": caja["class_name"],

                "xmin_normalized": caja["xmin"],
                "ymin_normalized": caja["ymin"],
                "xmax_normalized": caja["xmax"],
                "ymax_normalized": caja["ymax"],

                "xmin_pixels": round(
                    caja["xmin"]
                    * fila["image_width"]
                ),

                "ymin_pixels": round(
                    caja["ymin"]
                    * fila["image_height"]
                ),

                "xmax_pixels": round(
                    caja["xmax"]
                    * fila["image_width"]
                ),

                "ymax_pixels": round(
                    caja["ymax"]
                    * fila["image_height"]
                ),
            })


agregar_split(
    df_train,
    "TRAINING"
)

agregar_split(
    df_validation,
    "VALIDATION"
)

agregar_split(
    df_test,
    "TEST"
)


# ============================================================
# GUARDAR CSV SIN CABECERA PARA MODEL MAKER
# ============================================================

with open(
    CSV_MODELO,
    "w",
    newline="",
    encoding="utf-8"
) as archivo_csv:

    escritor = csv.writer(archivo_csv)

    escritor.writerows(
        filas_model_maker
    )


# ============================================================
# GUARDAR CSV LEGIBLE
# ============================================================

df_annotations = pd.DataFrame(
    filas_legibles
)

df_annotations.to_csv(
    CSV_LECTURA,
    index=False,
    encoding="utf-8-sig"
)


print(f"✅ CSV para Model Maker: {CSV_MODELO}")
print(f"✅ CSV legible: {CSV_LECTURA}")
print(f"Bounding boxes totales: {len(filas_model_maker)}")

display(
    df_annotations.head(10)
)


## Celda 10 — Generar data.yaml y archivos de clases


In [ ]:
# ============================================================
# CREAR data.yaml
# ============================================================

contenido_yaml = {
    "nc": len(NOMBRES_CLASES),
    "names": NOMBRES_CLASES,
}

with open(
    DATASET_PREPARADO / "data.yaml",
    "w",
    encoding="utf-8"
) as archivo_yaml:

    yaml.safe_dump(
        contenido_yaml,
        archivo_yaml,
        allow_unicode=True,
        sort_keys=False
    )


# ============================================================
# CREAR labels.txt
# ============================================================

with open(
    DATASET_PREPARADO / "labels.txt",
    "w",
    encoding="utf-8"
) as archivo_labels:

    for class_id in sorted(NOMBRES_CLASES):

        archivo_labels.write(
            NOMBRES_CLASES[class_id] + "\n"
        )


print("✅ data.yaml y labels.txt generados.")


## Celda 11 — Cargar el dataset en Model Maker


In [ ]:
# ============================================================
# CARGAR CSV EN MODEL MAKER
# ============================================================

train_data, validation_data, test_data = (
    object_detector.DataLoader.from_csv(
        str(CSV_MODELO)
    )
)


print("Train:", train_data.size)
print("Validation:", validation_data.size)
print("Test:", test_data.size)

if train_data.size == 0:
    raise RuntimeError("El conjunto train quedó vacío.")

if validation_data.size == 0:
    raise RuntimeError("El conjunto validation quedó vacío.")

if test_data.size == 0:
    raise RuntimeError("El conjunto test quedó vacío.")


## Celda 12 — Crear EfficientDet-Lite1


In [ ]:
# ============================================================
# SELECCIONAR EFFICIENTDET-LITE1
# ============================================================

spec = model_spec.get(
    "efficientdet_lite1"
)

print("✅ Arquitectura seleccionada: EfficientDet-Lite1")


## Celda 13 — Entrenar el detector


In [ ]:
# ============================================================
# ENTRENAR EFFICIENTDET-LITE1
# ============================================================

gc.collect()

modelo = object_detector.create(
    train_data=train_data,
    model_spec=spec,
    validation_data=validation_data,

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,

    train_whole_model=TRAIN_WHOLE_MODEL
)

print("✅ Entrenamiento finalizado.")


Model Maker gestiona internamente la configuración de entrenamiento de EfficientDet. En esta API no se expone un parámetro simple para sustituir directamente el optimizador por Adam en object_detector.create(). Conviene mantener el optimizador definido por la implementación oficial del modelo.


## Celda 14 — Evaluar el modelo TensorFlow


In [ ]:
# ============================================================
# EVALUAR MODELO ORIGINAL
# ============================================================

resultados_tensorflow = modelo.evaluate(
    test_data
)

print("Resultados del modelo TensorFlow:")
print(resultados_tensorflow)


with open(
    RESULTADOS_DIR / "evaluacion_tensorflow.txt",
    "w",
    encoding="utf-8"
) as archivo:

    archivo.write(
        str(resultados_tensorflow)
    )



Las métricas de evaluación que devuelve Model Maker corresponden al sistema de evaluación COCO.


## Celda 15 — Exportar TFLite Float16

La exportación con QuantizationConfig.for_float16() es el método oficial para generar EfficientDet-Lite con pesos Float16. El modelo exportado incluye metadatos y el mapa de etiquetas.


In [ ]:
# ============================================================
# EXPORTAR TFLITE FLOAT16
# ============================================================

configuracion_fp16 = (
    QuantizationConfig.for_float16()
)

modelo.export(
    export_dir=str(EXPORT_DIR),
    tflite_filename=MODELO_FP16.name,
    quantization_config=configuracion_fp16
)

if not MODELO_FP16.exists():

    raise RuntimeError(
        "No se generó el modelo TFLite Float16."
    )


print("✅ Modelo Float16 generado:")
print(MODELO_FP16)

print(
    "Tamaño:",
    f"{MODELO_FP16.stat().st_size / 1024 / 1024:.2f} MB"
)


## Celda 16 — Exportar también una versión INT8

Para Android con CPU, INT8 puede ser más rápido. Float16 suele ser una buena alternativa cuando se utiliza aceleración GPU. Esta celda genera ambas versiones para que pueda compararlas.


In [ ]:
# ============================================================
# EXPORTAR VERSIÓN CUANTIZADA POR DEFECTO
# GENERALMENTE INT8
# ============================================================

modelo.export(
    export_dir=str(EXPORT_DIR),
    tflite_filename=MODELO_INT8.name
)

print("✅ Modelo INT8 generado:")
print(MODELO_INT8)

print(
    "Tamaño:",
    f"{MODELO_INT8.stat().st_size / 1024 / 1024:.2f} MB"
)


## Celda 17 — Evaluar el TFLite Float16

La documentación advierte que la conversión TFLite puede modificar el comportamiento del posprocesamiento y el número máximo de detecciones, por lo que conviene evaluar también el archivo exportado.


In [ ]:
# ============================================================
# EVALUAR TFLITE FLOAT16
# ============================================================

resultados_tflite = modelo.evaluate_tflite(
    str(MODELO_FP16),
    test_data
)

print("Resultados TFLite Float16:")
print(resultados_tflite)


with open(
    RESULTADOS_DIR / "evaluacion_tflite_fp16.txt",
    "w",
    encoding="utf-8"
) as archivo:

    archivo.write(
        str(resultados_tflite)
    )


## Celda 18 — Verificar entrada y salidas del modelo


In [ ]:
# ============================================================
# VERIFICAR MODELO TFLITE
# ============================================================

interpreter = tf.lite.Interpreter(
    model_path=str(MODELO_FP16)
)

interpreter.allocate_tensors()

detalles_entrada = (
    interpreter.get_input_details()
)

detalles_salida = (
    interpreter.get_output_details()
)


print("=" * 60)
print("ENTRADA")
print("=" * 60)

for detalle in detalles_entrada:

    print("Nombre:", detalle["name"])
    print("Forma:", detalle["shape"])
    print("Tipo:", detalle["dtype"])
    print()


print("=" * 60)
print("SALIDAS")
print("=" * 60)

for detalle in detalles_salida:

    print("Nombre:", detalle["name"])
    print("Forma:", detalle["shape"])
    print("Tipo:", detalle["dtype"])
    print()


## Celda 19 — Preparar paquete Android


In [ ]:
# ============================================================
# CREAR PAQUETE PARA ANDROID
# ============================================================

if PAQUETE_ANDROID.exists():
    shutil.rmtree(PAQUETE_ANDROID)

PAQUETE_ANDROID.mkdir(
    parents=True,
    exist_ok=True
)


# Copiar modelos
shutil.copy2(
    MODELO_FP16,
    PAQUETE_ANDROID / MODELO_FP16.name
)

if MODELO_INT8.exists():

    shutil.copy2(
        MODELO_INT8,
        PAQUETE_ANDROID / MODELO_INT8.name
    )


# Copiar archivos complementarios
archivos_complementarios = [
    DATASET_PREPARADO / "labels.txt",
    DATASET_PREPARADO / "data.yaml",
    CSV_LECTURA,
    RESULTADOS_DIR / "evaluacion_tensorflow.txt",
    RESULTADOS_DIR / "evaluacion_tflite_fp16.txt",
]

for archivo in archivos_complementarios:

    if archivo.exists():

        shutil.copy2(
            archivo,
            PAQUETE_ANDROID / archivo.name
        )


# ============================================================
# CREAR INFORMACIÓN DEL MODELO
# ============================================================

informacion = {
    "architecture": "EfficientDet-Lite1",
    "framework": "TensorFlow Lite Model Maker",
    "class_count": len(NOMBRES_CLASES),
    "classes": NOMBRES_CLASES,

    "float16_model": MODELO_FP16.name,
    "int8_model": (
        MODELO_INT8.name
        if MODELO_INT8.exists()
        else None
    ),

    "train_images": len(df_train),
    "validation_images": len(df_validation),
    "test_images": len(df_test),

    "bounding_boxes": len(filas_model_maker),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,

    "android_api": (
        "TensorFlow Lite Task Library ObjectDetector"
    ),
}


with open(
    PAQUETE_ANDROID / "modelo_info.json",
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        informacion,
        archivo,
        indent=4,
        ensure_ascii=False
    )


# ============================================================
# CREAR README
# ============================================================

readme = f"""
MODELO EFFICIENTDET-LITE1 PARA ANDROID
======================================

Clase:
- botellas

Modelo Float16:
- {MODELO_FP16.name}

Modelo INT8:
- {MODELO_INT8.name}

Uso recomendado:
- Float16: GPU Delegate
- INT8: CPU

Copiar el archivo .tflite elegido a:

app/src/main/assets/

El modelo contiene metadatos y etiquetas para utilizarlo
con TensorFlow Lite Task Library ObjectDetector.
""".strip()


with open(
    PAQUETE_ANDROID / "README.txt",
    "w",
    encoding="utf-8"
) as archivo:

    archivo.write(readme)


print("✅ Paquete Android preparado.")


## Celda 20 — Comprimir y descargar


In [ ]:
# ============================================================
# CREAR ZIP FINAL
# ============================================================

shutil.make_archive(
    base_name=str(
        ZIP_FINAL.with_suffix("")
    ),
    format="zip",
    root_dir=PAQUETE_ANDROID
)


print("=" * 65)
print("PROCESO FINALIZADO")
print("=" * 65)

print(f"ZIP: {ZIP_FINAL.name}")

print(
    "Tamaño del ZIP:",
    f"{ZIP_FINAL.stat().st_size / 1024 / 1024:.2f} MB"
)


# ============================================================
# DESCARGAR AL DISCO
# ============================================================

files.download(
    str(ZIP_FINAL)
)


Resultado descargado

El ZIP final incluirá:

efficientdet_lite1_botellas_android.zip
├── efficientdet_lite1_botellas_fp16.tflite
├── efficientdet_lite1_botellas_int8.tflite
├── labels.txt
├── data.yaml
├── annotations_legible.csv
├── modelo_info.json
├── evaluacion_tensorflow.txt
├── evaluacion_tflite_fp16.txt
└── README.txt
Uso básico en Android

Coloque el modelo en:

app/src/main/assets/efficientdet_lite1_botellas_fp16.tflite

La API ObjectDetector de TensorFlow Lite Task Library procesa el redimensionamiento, la rotación, el umbral de confianza y el mapa de etiquetas. Los modelos creados con Model Maker están incluidos entre los modelos oficialmente compatibles.

Dependencia:

dependencies {
    implementation "org.tensorflow:tensorflow-lite-task-vision"
    implementation "org.tensorflow:tensorflow-lite-gpu-delegate-plugin"
}

Inicialización en Kotlin:

import org.tensorflow.lite.task.core.BaseOptions
import org.tensorflow.lite.task.vision.detector.ObjectDetector

val baseOptions = BaseOptions.builder()
    .useGpu()
    .build()

val options = ObjectDetector.ObjectDetectorOptions.builder()
    .setBaseOptions(baseOptions)
    .setScoreThreshold(0.35f)
    .setMaxResults(20)
    .build()

val detector = ObjectDetector.createFromFileAndOptions(
    context,
    "efficientdet_lite1_botellas_fp16.tflite",
    options
)

Para celulares donde el GPU Delegate dé problemas, utilice el modelo INT8 y elimine .useGpu() de la configuración.
